In [1]:
import torch

from torch import nn
from torch.utils.data import DataLoader

from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights
from torchvision import transforms

from tqdm import tqdm

from utils import BaseImageFolderDataset, accuracy

In [2]:
GPU = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
BATCH_SIZE = 128
IMAGENET_CLASSES = ResNet50_Weights.IMAGENET1K_V2.meta['categories']

In [3]:
class ImageNetSketch(BaseImageFolderDataset):
    URL = 'https://www.kaggle.com/api/v1/datasets/download/wanghaohan/imagenetsketch'
    ARCHIVE_NAME = 'ImageNet-Sketch.zip'
    EXTRACTED_FOLDER = 'imagenet-sketch/sketch'

In [4]:
class VisDA2017(BaseImageFolderDataset):
    URL = 'http://csr.bu.edu/ftp/visda17/clf/train.tar'
    ARCHIVE_NAME = 'train.tar'
    EXTRACTED_FOLDER = 'train'

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.to_imagenet = {
            0: 726,  # aeroplane
            1: 444,  # bicycle
            2: 654,  # bus
            3: 705,  # car
            4: 603,  # hourse
            5: 0,    # knife (absent)
            6: 444,  # motorcycle (absent)
            7: 0,    # man (absent)
            8: 985,  # plant
            9: 667,  # skateboard (absent)
            10: 466, # train
            11: 867  # truck
        }

    def __getitem__(self, idx: int):
        image, label = super().__getitem__(idx)

        return image, self.to_imagenet[label]

In [5]:
def get_resnet_classname(probs: torch.Tensor) -> list[str]:
    categories = ResNet50_Weights.IMAGENET1K_V2.meta['categories']
    
    if probs.dim == 1:
        probs = probs.unsqueeze(0)
    
    return [categories[idx] for idx in probs.argmax(dim=1)]

In [6]:
def test(model: nn.Module,
         dataloader: DataLoader,
         device: torch.device = torch.device('cpu')) -> tuple[float, float]:
    acc1 = 0
    acc5 = 0

    model.eval()

    with torch.no_grad():
        for x_batch, y_batch in tqdm(dataloader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(x_batch)

            acc = accuracy(pred, y_batch)

            acc1 += float(acc[0])
            acc5 += float(acc[1])

    acc1 /= len(dataloader)
    acc5 /= len(dataloader)

    return acc1, acc5

In [7]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [8]:
ins_dataset = ImageNetSketch('./data', transform=preprocess, download=True)
vis_dataset = VisDA2017('./data', transform=preprocess, download=True)

In [9]:
len(ins_dataset), len(vis_dataset)

(50889, 152397)

In [10]:
ins_dataloader = DataLoader(
    dataset=ins_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True, 
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4
)

In [11]:
vis_dataloader = DataLoader(
    dataset=vis_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4
)

In [12]:
resnet18_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).to(GPU)

In [13]:
resnet50_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(GPU)

In [14]:
resnet18_acc = test(resnet18_model, ins_dataloader, GPU)
resnet50_acc = test(resnet50_model, ins_dataloader, GPU)

print(
    f'resnet18 | acc@1: {resnet18_acc[0]:.1f} | acc@5: {resnet18_acc[1]:.1f}')
print(
    f'resnet50 | acc@1: {resnet50_acc[0]:.1f} | acc@5: {resnet50_acc[1]:.1f}')

100%|██████████| 398/398 [05:30<00:00,  1.21it/s]

resnet18 | acc@1: 20.2 | acc@5: 37.3
resnet50 | acc@1: 28.4 | acc@5: 46.5


| Model    | acc@1 | acc@5 |
|----------|-------|-------|
| ResNet18 | 20.2  | 37.3  |
| ResNet50 | 28.4  | 46.6  |

In [15]:
torch.cuda.empty_cache()

In [14]:
resnet18_acc = test(resnet18_model, vis_dataloader, GPU)
resnet50_acc = test(resnet50_model, vis_dataloader, GPU)

print(
    f'resnet18 | acc@1: {resnet18_acc[0]:.1f} | acc@5: {resnet18_acc[1]:.1f}')
print(
    f'resnet50 | acc@1: {resnet50_acc[0]:.1f} | acc@5: {resnet50_acc[1]:.1f}')

100%|██████████| 1191/1191 [16:11<00:00,  1.23it/s]

resnet18 | acc@1: 1.2 | acc@5: 8.4
resnet50 | acc@1: 2.0 | acc@5: 9.3


So, It was expected.